# Les 08 - Multi-Agent Ontwerppatroon


## Installatie


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Waarom Multi-Agent Systemen?

Taken uit de echte wereld, zoals het plannen van een reis, vereisen veel verschillende soorten expertise — logistiek, lokale kennis, budgettering, en meer. Een enkele agent die probeert alles te regelen, wordt al snel onhandelbaar.

Multi-agent systemen lossen dit op door **specialisatie**: elke agent richt zich op een specifiek deskundigheidsgebied, wat resulteert in hogere kwaliteit dan een generalist. Ze verbeteren ook de **schaalbaarheid** — je kunt nieuwe agenten toevoegen (bijvoorbeeld een vluchtspecialist, een restaurantrecensent) zonder de bestaande workflow te herschrijven. De agenten werken samen via een gestructureerde pijplijn, waarbij ze context van de ene naar de andere doorgeven.


## Gespecialiseerde Agenten Creëren


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Een sequentiële workflow bouwen

`WorkflowBuilder` stelt je in staat om agenten aan te sluiten in een gerichte grafiek. Hier maken we een eenvoudige tweestaps pijplijn: de **TravelPlanner** stelt de reisroute op, daarna beoordeelt en verbetert de **TravelConcierge** deze.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Meer Agenten Toevoegen aan de Workflow

Een van de grootste voordelen van het multi-agent patroon is hoe eenvoudig het is om uit te breiden. Hieronder voegen we een **BudgetReviewer** agent toe die het plan controleert aan de hand van het budget van de reiziger, items markeert die de kosten mogelijk boven de limiet brengen, en geldbesparende alternatieven suggereert. De workflow draait nu drie agenten achter elkaar:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Samenvatting

In deze les heb je geleerd hoe je:

1. **Gespecialiseerde agenten maakt** — elk met een gerichte rol (planning, conciërge, budgetcontrole).
2. **Agenten aansluit in een sequentiële workflow** met behulp van `WorkflowBuilder` en `add_edge`.
3. **Output streamt** vanuit een multi-agent pijplijn, waarbij wordt bijgehouden welke agent aan het spreken is.
4. **Een workflow uitbreidt** door nieuwe agenten aan de keten toe te voegen zonder bestaande te wijzigen.

Het multi-agent ontwerppatroon houdt elke agent eenvoudig terwijl het rijkere, grondiger beoordeelde resultaten oplevert dan enig enkele agent alleen zou kunnen bereiken.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:  
Dit document is vertaald met behulp van de AI-vertalingsservice [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat automatische vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het oorspronkelijke document in de originele taal moet als de gezaghebbende bron worden beschouwd. Voor cruciale informatie wordt een professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties voortvloeiend uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
